# conv-kernel-shape — ex1: introspect a conv2d weight tensor's axes

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `conv-kernel-shape`. Running the final beacon cell reports progress against the `CNN: Kernel shape (OC, IC, KH, KW)` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `CNN: Kernel shape (OC, IC, KH, KW)` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`conv-kernel-shape`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "conv-kernel-shape"
DD_SUBTOPIC = "CNN: Kernel shape (OC, IC, KH, KW)"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Conv kernel shape `(OC, IC, KH, KW)` — quick refresher

`nn.Conv2d(in_channels=IC, out_channels=OC, kernel_size=(KH, KW)).weight` has shape **`(OC, IC, KH, KW)`** — output channels first. This is the PyTorch convention and matches the einsum role of each axis:

```
weight[oc, ic, kh, kw]  →  the (kh, kw) tap of input-channel ic
                            for output-channel oc
```

**Why `OC` is first.** A `nn.Conv2d` is a stack of `OC` independent filters, each shaped `(IC, KH, KW)`. Listing `OC` as the leading axis makes `weight[i]` directly index the `i`-th filter — convenient for visualization or per-filter analysis.

**Common confusion.** `nn.ConvTranspose2d` flips this to `(IC, OC, KH, KW)` (see the `convT-kernel-axis-swap` atom). And `nn.Linear` uses `(out_features, in_features)` — same OC-first convention as Conv2d but with no spatial axes. Mixing these up at init time produces silently wrong models; mixing them at forward time produces a shape error.

**Parameter count.** Total params (no bias) = `OC * IC * KH * KW`. Bias adds `OC` more. A 3×3 conv from 64→128 channels is `128 * 64 * 9 = 73,728` weights — most of a CNN's storage lives here.

### Exercise 1 — introspect a conv2d weight tensor's axes

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply the `(OC, IC, KH, KW)` weight-shape convention to extract the four axes of a `nn.Conv2d` weight tensor and compute the exact parameter count (weights only, no bias).
> Keywords: conv2d, weight-shape, introspection, param-count
> ```

**KCs targeted:** `conv-kernel-axis-order`, `conv-param-count-from-shape`

Implement `ex1_conv2d_weight_facts(conv)`. Given an instantiated `nn.Conv2d` module, return a dict with the following keys:

- `'out_channels'` (int) — `OC`, first axis of `weight`.
- `'in_channels'`  (int) — `IC`, second axis.
- `'kernel_height'` (int) — `KH`, third axis.
- `'kernel_width'`  (int) — `KW`, fourth axis.
- `'n_weight_params'` (int) — total scalars in `weight` (`OC * IC * KH * KW`).

**Hint.** Read `conv.weight.shape` and unpack the four axes. Do NOT trust `conv.in_channels` / `conv.out_channels` — derive everything from `weight.shape` so the drill targets the layout, not the module's stored attributes.

The test instantiates several `nn.Conv2d` modules and confirms your extraction matches the constructor args.

In [ ]:
def ex1_conv2d_weight_facts(conv) -> dict:
    OC, IC, KH, KW = conv.weight.shape
    return {
        'out_channels':    int(OC),
        'in_channels':     int(IC),
        'kernel_height':   int(KH),
        'kernel_width':    int(KW),
        'n_weight_params': int(OC * IC * KH * KW),
    }


<details><summary>Solution</summary>

```python
def ex1_conv2d_weight_facts(conv) -> dict:
    OC, IC, KH, KW = conv.weight.shape
    return {
        'out_channels':    int(OC),
        'in_channels':     int(IC),
        'kernel_height':   int(KH),
        'kernel_width':    int(KW),
        'n_weight_params': int(OC * IC * KH * KW),
    }
```

**Why `int(...)` casts.** `conv.weight.shape` is a `torch.Size`, whose entries are plain `int` already in modern PyTorch — but on older builds they can be `torch.SymInt` (under compile/dynamic shapes). Explicit `int()` guarantees JSON-serializable plain ints regardless of source.

**Why NOT use `conv.in_channels`.** The drill targets the *layout convention* — i.e., the fact that `weight.shape[0] == OC`. Reading `conv.in_channels` would bypass that and miss the point. The exception (out of scope for this drill) is `nn.ConvTranspose2d`, where the same `.in_channels` attribute exists but `weight.shape[0] == IC` (the swapped layout).

**Param-count consequence.** A typical ResNet-block conv 256→512 kernel 3×3 is 1.18M params. Multiply by the dozens of such layers in a deep model and you see why CNNs are mostly *kernel weights* — far more than the input data itself.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()